In [1]:
!pip list

Package                      Version
---------------------------- ------------
absl-py                      2.1.0
affine                       2.4.0
asttokens                    2.4.1
astunparse                   1.6.3
attrs                        25.1.0
certifi                      2024.7.4
charset-normalizer           3.3.2
click                        8.1.8
click-plugins                1.1.1
cligj                        0.7.2
colorama                     0.4.6
comm                         0.2.2
contourpy                    1.2.1
cycler                       0.12.1
debugpy                      1.8.2
decorator                    5.1.1
exceptiongroup               1.2.1
executing                    2.0.1
filelock                     3.15.4
flatbuffers                  24.3.25
fonttools                    4.53.1
fsspec                       2024.6.1
future                       1.0.0
gast                         0.6.0
google-pasta                 0.2.0
grpcio                       1.64.

In [ ]:
#big
import serial
import time

PORT = 'COM8'  # ← your actual port
BAUDRATE = 115200

start_cmd = bytes([0xFA, 0x01, 0xFF, 0x04, 0x01, 0x00, 0x00, 0x00, 0x00, 0xFF])
stop_cmd  = bytes([0xFA, 0x01, 0xFF, 0x04, 0x00, 0x00, 0x00, 0x00, 0x00, 0xFE])

ser = serial.Serial(PORT, BAUDRATE, timeout=1)
time.sleep(1)

ser.write(start_cmd)
print("Started continuous measurement. Press Ctrl+C to stop.")

try:
    while True:
        if ser.in_waiting >= 9:
            data = ser.read(9)
            if data[0] == 0xFB and data[1] == 0x03:
                valid_flag = int.from_bytes(data[4:6], 'little')
                distance_dm = int.from_bytes(data[6:8], 'little')
                
                if valid_flag == 1:
                    print(f"Distance: {distance_dm / 10:.1f} meters")
                else:
                    print("Invalid reading.")
        time.sleep(0.1)

except KeyboardInterrupt:
    print("\nStopping...")
    ser.write(stop_cmd)
    ser.close()
    print("Sensor stopped and port closed.")

Started continuous measurement. Press Ctrl+C to stop.
Invalid reading.
Invalid reading.

Stopping...
Sensor stopped and port closed.


In [ ]:
#small LRF
import serial
import time

PORT = 'COM9'
BAUDRATE = 115200

start_cmd = bytes([0xFA, 0x01, 0xFF, 0x04, 0x01, 0x00, 0x00, 0x00, 0x00, 0xFF])
stop_cmd  = bytes([0xFA, 0x01, 0xFF, 0x04, 0x00, 0x00, 0x00, 0x00, 0x00, 0xFE])

ser = serial.Serial(PORT, BAUDRATE, timeout=1)
time.sleep(1)

ser.write(start_cmd)
print("Started continuous measurement. Press Ctrl+C to stop.")

buffer = bytearray()

try:
    while True:
        if ser.in_waiting:
            buffer += ser.read(ser.in_waiting)

            # Look for valid frame (starts with fb 03, total 9 bytes)
            while len(buffer) >= 9:
                if buffer[0] == 0xFB and buffer[1] == 0x03:
                    frame = buffer[:9]
                    buffer = buffer[9:]  # remove the used frame

                    valid_flag = int.from_bytes(frame[4:6], 'little')
                    distance_dm = int.from_bytes(frame[6:8], 'little')

                    if valid_flag == 1:
                        print(f"Distance: {distance_dm / 10:.1f} meters")
                    else:
                        print("Invalid reading.")
                else:
                    # Skip until we find start of frame
                    buffer = buffer[1:]

        time.sleep(0.01)

except KeyboardInterrupt:
    print("\nStopping...")
    ser.write(stop_cmd)
    ser.close()
    print("Sensor stopped and port closed.")

SerialException: could not open port 'COM9': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)

Fuse this two code into a single cell 
Print: 

Far LRF:
Near LRF:

Fuse with latest camera code from job